# Actividad — Comparación de OLS, Ridge y Lasso en alta dimensión
**Aprendizaje Supervisado · Sesión 5 · Regresión regularizada**

**Qué haremos.** En la presentación vimos que, cuando hay muchas variables y pocos datos, OLS memoriza el ruido, y que
**Ridge** (L2), **Lasso** (L1) y **Elastic Net** frenan al modelo penalizando los pesos grandes. Hoy los comparas **con el mismo protocolo** en un
conjunto de **alta dimensión** simulado (100 observaciones de entrenamiento, 150 variables, solo 10 realmente relevantes): ¿cuánto sobreajusta OLS?
¿qué hace cada penalización con los coeficientes? ¿qué método predice mejor y cuál recupera las variables correctas? ¿cambia la respuesta si la verdad no es dispersa?

| Parte | Pregunta que responde | Tema de la presentación |
|---|---|---|
| 0–1 | ¿Qué datos tenemos y cuánto sobreajusta OLS? | A1 |
| 2 | ¿Cómo se ven las curvas de aprendizaje y qué pasa cerca de $n=p$? | A2–A4 |
| 3 | ¿Qué hace Ridge con los coeficientes y cómo se elige $\alpha$? | B1–B5 |
| 4 | ¿Qué hace Lasso y qué variables conserva? | C1–C6 |
| 5 | ¿Qué aporta Elastic Net con variables correlacionadas? | D1–D4 |
| 6 | ¿Cómo se comparan los cuatro modelos? | E1–E3 |
| 7 | ¿Y si todos los coeficientes importan un poco? | E4 |
| 8 | Recomendación final | E5 |

**Tiempo estimado:** 60–75 minutos de trabajo individual.

**Entregable:** este notebook ejecutado (con tus respuestas en las celdas ✏️) y el archivo `entregable_unidad5.csv`
con la tabla *método · hiperparámetros · RMSE de entrenamiento · RMSE de prueba · variables usadas · reales halladas · comentario* de la Parte 6.

**Cómo trabajar.** Ejecuta las celdas en orden. Donde dice ✏️ hay una línea de código por completar o una pregunta;
escribe tus respuestas en la celda de texto que sigue, bajo «**Respuesta:**».
Los datos son **simulados**: como conocemos la verdad (qué variables importan de verdad), podemos medir si cada método la recupera.

> ⚠️ Todos los modelos llevan `StandardScaler` **dentro** del `Pipeline`: la penalización trata igual a todos los pesos y por eso las variables deben estar en la misma escala.

## Parte 0 — Los datos

Cada observación tiene **150 variables**. Solo las primeras 10 afectan a $y$: $x_1,x_2,x_3$ valen $+2$, $x_4,x_5,x_6$ valen $-1.5$ y $x_7,\dots,x_{10}$ valen $3,-2,1.5,1$. Las otras 140 son ruido puro. Además, $x_1$–$x_3$ son tres copias ruidosas de una misma señal (correlación $\approx 0.8$); igual $x_4$–$x_6$. A $y$ se le suma ruido de desviación $\sigma=3$, de modo que **ningún modelo puede tener RMSE menor que 3**.

In [ ]:
import warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet,
                                  RidgeCV, LassoCV, ElasticNetCV, lasso_path)
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

P, SIGMA = 150, 3.0
BETA = np.zeros(P)
BETA[[0, 1, 2]] = 2.0
BETA[[3, 4, 5]] = -1.5
BETA[[6, 7, 8, 9]] = [3, -2, 1.5, 1]


def genera(n, semilla, beta=BETA):
    # Devuelve (X_entrenamiento, y_entrenamiento, X_prueba, y_prueba); la prueba siempre tiene 3000 filas
    rng = np.random.default_rng(semilla)

    def una(nn):
        X = rng.standard_normal((nn, P))
        L1, L2 = rng.standard_normal(nn), rng.standard_normal(nn)
        for j in range(3):
            X[:, j] = 0.9 * L1 + np.sqrt(0.19) * X[:, j]
        for j in range(3, 6):
            X[:, j] = 0.9 * L2 + np.sqrt(0.19) * X[:, j]
        return X

    Xa = una(n);  ya = Xa @ beta + SIGMA * rng.standard_normal(n)
    Xb = una(3000); yb = Xb @ beta + SIGMA * rng.standard_normal(3000)
    return Xa, ya, Xb, yb


def rmse(modelo, X, y):
    return mean_squared_error(y, modelo.predict(X)) ** 0.5


kf = KFold(5, shuffle=True, random_state=0)       # protocolo de validación que usaremos SIEMPRE
ALPHAS = np.logspace(-3, 1.2, 60)                 # rejilla de alfas para Lasso y Elastic Net

Xtr, ytr, Xte, yte = genera(100, semilla=4)
print("Entrenamiento:", Xtr.shape, "  Prueba:", Xte.shape)
print("Variables realmente relevantes (índices):", np.flatnonzero(BETA))
print("Correlación entre x1 y x2:", round(np.corrcoef(Xtr[:, 0], Xtr[:, 1])[0, 1], 2),
      "  entre x1 y x50:", round(np.corrcoef(Xtr[:, 0], Xtr[:, 49])[0, 1], 2))

## Parte 1 — OLS en alta dimensión  *(presentación A1)*

Ajustamos mínimos cuadrados ordinarios con las 150 variables estandarizadas y comparamos el error en entrenamiento con el error en datos nuevos. Con $p>n$ hay más incógnitas que datos; `LinearRegression` toma entonces la solución de menor norma.

In [ ]:
ols = ...     # ✏️ make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)
print(f"OLS  RMSE entrenamiento = {rmse(ols, Xtr, ytr):.2f}    RMSE prueba = {rmse(ols, Xte, yte):.2f}")
print(f"Piso teórico (ruido) = {SIGMA}")
print("Coeficiente más grande en valor absoluto:", round(np.abs(ols[-1].coef_).max(), 2))

✏️ **Pregunta 1.1.** ¿Cuántas incógnitas (pesos) y cuántas observaciones hay? ¿Qué indica un RMSE de entrenamiento casi 0 junto a un RMSE de prueba mucho mayor que 3? ¿Es un caso de subajuste o de sobreajuste?

**Respuesta:**

## Parte 2 — Curvas de aprendizaje  *(presentación A2–A4)*

Repetimos el ajuste con conjuntos de entrenamiento de tamaño $n$ creciente (promediando 4 muestras por tamaño) y medimos el error en entrenamiento y en prueba.

In [ ]:
tamanos = [40, 70, 100, 130, 150, 170, 220, 300, 500, 900]
filas = []
for n in tamanos:
    for r in range(4):
        X1, y1, X2, y2 = genera(n, semilla=200 + r)
        m = ...     # ✏️ make_pipeline(StandardScaler(), LinearRegression()).fit(X1, y1)
        filas.append({"n": n, "entrenamiento": rmse(m, X1, y1), "prueba": rmse(m, X2, y2)})
curva = pd.DataFrame(filas).groupby("n").mean()
print(curva.round(2).T.to_string())

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(curva.index, curva["entrenamiento"], "o--", label="entrenamiento")
ax.plot(curva.index, curva["prueba"].clip(upper=15), "o-", label="prueba (cortada en 15)")
ax.axhline(SIGMA, color="k", ls=":", lw=1); ax.axvline(P, color="C3", ls="--", lw=1)
ax.set_xscale("log"); ax.set_xlabel("n (tamaño de entrenamiento), con p = 150"); ax.set_ylabel("RMSE"); ax.legend(); plt.tight_layout(); plt.show()

✏️ **Pregunta 2.1.** (a) ¿Qué ocurre con el error de entrenamiento y con el de prueba de OLS cuando $n$ crece? (b) ¿Qué pasa exactamente alrededor de $n=p=150$? (c) Con $n=900$, ¿qué tan cerca está OLS del piso de 3?

**Respuesta:**

Ahora agregamos **Ridge** y **Lasso** (con $\alpha$ elegido por validación cruzada en cada muestra) a la curva de prueba. Esta celda ya está completa: solo ejecútala y observa.

In [ ]:
res = {"OLS": [], "Ridge": [], "Lasso": []}
for n in tamanos:
    acum = {k: [] for k in res}
    for r in range(4):
        X1, y1, X2, y2 = genera(n, semilla=200 + r)
        mods = {"OLS": make_pipeline(StandardScaler(), LinearRegression()),
                "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 50), cv=kf)),
                "Lasso": make_pipeline(StandardScaler(), LassoCV(alphas=ALPHAS, cv=kf, max_iter=100000))}
        for k, m in mods.items():
            acum[k].append(rmse(m.fit(X1, y1), X2, y2))
    for k in res:
        res[k].append(np.mean(acum[k]))
tabla_curvas = pd.DataFrame(res, index=tamanos); tabla_curvas.index.name = "n"
print(tabla_curvas.round(2).T.to_string())

fig, ax = plt.subplots(figsize=(6.5, 3.4))
for k in res:
    ax.plot(tamanos, np.minimum(res[k], 15), "o-", ms=4, label=k)
ax.axhline(SIGMA, color="k", ls=":", lw=1); ax.axvline(P, color="C3", ls="--", lw=1)
ax.set_xscale("log"); ax.set_xlabel("n (tamaño de entrenamiento), con p = 150"); ax.set_ylabel("RMSE de prueba (cortado en 15)"); ax.legend(); plt.tight_layout(); plt.show()

✏️ **Pregunta 2.2.** (a) ¿Cuándo es más grande la ventaja de regularizar sobre OLS: con $n$ muy chico respecto a $p$ o con $n$ muy grande? (b) ¿Qué modelo se acerca más al piso de 3 con $n=100$ y por qué crees que le conviene más este problema?

**Respuesta:**

## Parte 3 — Ridge (L2)  *(presentación B1–B5)*

Estandarizamos una vez (solo para trazar la trayectoria; en lo demás el escalador irá dentro del `Pipeline`) y ajustamos Ridge para una rejilla de $\alpha$.

In [ ]:
sc = StandardScaler().fit(Xtr); Xs = sc.transform(Xtr)
alphas_r = np.logspace(-2, 5, 70)
coefs_r = []
for a in alphas_r:
    m = ...     # ✏️ Ridge(alpha=a).fit(Xs, ytr)
    coefs_r.append(m.coef_)
coefs_r = np.array(coefs_r)

for a in [0.01, 1, 100, 10000]:
    i = int(np.argmin(np.abs(alphas_r - a)))
    print(f"alpha = {alphas_r[i]:9.2f}:  norma de los pesos = {np.linalg.norm(coefs_r[i]):6.2f}   pesos exactamente 0: {(coefs_r[i] == 0).sum()}")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
for j in range(10, P):
    ax.plot(alphas_r, coefs_r[:, j], color="#B0BEC5", lw=0.6)
for j in range(10):
    ax.plot(alphas_r, coefs_r[:, j], color="C0" if j < 6 else "C1", lw=1.4)
ax.set_xscale("log"); ax.set_xlabel("alpha"); ax.set_ylabel("coeficiente (estandarizado)"); plt.tight_layout(); plt.show()

Ahora elegimos $\alpha$ con validación cruzada. El escalador va **dentro** del pipeline, así que se rehace en cada partición.

In [ ]:
alphas_cv = np.logspace(-2, 4, 25)
rm_cv, rm_tr = [], []
for a in alphas_cv:
    m = ...     # ✏️ make_pipeline(StandardScaler(), Ridge(alpha=a))
    rm_cv.append(-cross_val_score(m, Xtr, ytr, cv=kf, scoring="neg_root_mean_squared_error").mean())
    rm_tr.append(rmse(m.fit(Xtr, ytr), Xtr, ytr))
i = int(np.argmin(rm_cv))
print(f"Mejor alpha de la rejilla: {alphas_cv[i]:.1f}   RMSE de validación = {rm_cv[i]:.2f}   (con alpha = 0.01: {rm_cv[0]:.2f})")

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(alphas_cv, rm_tr, "--", label="entrenamiento"); ax.plot(alphas_cv, rm_cv, "o-", ms=3, label="validación cruzada")
ax.plot(alphas_cv[i], rm_cv[i], "k*", ms=12); ax.set_xscale("log"); ax.set_xlabel("alpha"); ax.set_ylabel("RMSE"); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
ridge = ...     # ✏️ make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 50), cv=kf)).fit(Xtr, ytr)
print(f"RidgeCV eligió alpha = {ridge[-1].alpha_:.1f}")
print(f"RMSE entrenamiento = {rmse(ridge, Xtr, ytr):.2f}    RMSE prueba = {rmse(ridge, Xte, yte):.2f}    (OLS en prueba: {rmse(ols, Xte, yte):.2f})")
print("Pesos distintos de cero:", int((ridge[-1].coef_ != 0).sum()), "de", P)

✏️ **Pregunta 3.1.** (a) Describe qué ocurre con los coeficientes cuando $\alpha$ crece; ¿llegan a ser exactamente cero? (b) ¿Cómo es la curva de validación (forma, mínimo) y qué pasa con el error de entrenamiento al subir $\alpha$? (c) ¿Cuánto mejora Ridge a OLS en prueba y cuántas variables usa?

**Respuesta:**

**Dos columnas casi idénticas.** Repetimos 200 veces un problema con dos variables que son casi la misma ($x_2=x_1+$ un poco de ruido) y $y=x_1+x_2+$ ruido. Los pesos verdaderos son $(1,1)$.

In [ ]:
rng = np.random.default_rng(1)
W_ols, W_ridge = [], []
for _ in range(200):
    x1 = rng.standard_normal(30); x2 = x1 + 0.05 * rng.standard_normal(30)
    Xd = np.c_[x1, x2]; yd = x1 + x2 + 0.5 * rng.standard_normal(30)
    W_ols.append(LinearRegression().fit(Xd, yd).coef_)
    W_ridge.append(...)     # ✏️ Ridge(alpha=1).fit(Xd, yd).coef_
W_ols, W_ridge = np.array(W_ols), np.array(W_ridge)
print("OLS   : peso de x1 -> promedio {:.2f}, desviación {:.2f};  suma w1+w2 -> promedio {:.2f}, desviación {:.2f}".format(
      W_ols[:, 0].mean(), W_ols[:, 0].std(), W_ols.sum(1).mean(), W_ols.sum(1).std()))
print("Ridge : peso de x1 -> promedio {:.2f}, desviación {:.2f};  suma w1+w2 -> promedio {:.2f}, desviación {:.2f}".format(
      W_ridge[:, 0].mean(), W_ridge[:, 0].std(), W_ridge.sum(1).mean(), W_ridge.sum(1).std()))

✏️ **Pregunta 3.2.** ¿Qué pesos individuales da OLS comparado con Ridge? ¿Qué tiene de estable la suma $w_1+w_2$ aun en OLS? Explica con el costo $w_1^2+w_2^2$ por qué Ridge prefiere repartir el peso.

**Respuesta:**

## Parte 4 — Lasso (L1)  *(presentación C1–C6)*

`lasso_path` calcula la trayectoria completa de coeficientes para una rejilla decreciente de $\alpha$ (usa variables estandarizadas y $y$ centrada).

In [ ]:
alphas_l, coefs_l, _ = ...     # ✏️ lasso_path(Xs, ytr - ytr.mean(), alphas=np.logspace(1.2, -3, 100))
nz = (np.abs(coefs_l) > 1e-8).sum(axis=0)
for a in [10, 1, 0.3, 0.1, 0.01]:
    i = int(np.argmin(np.abs(alphas_l - a)))
    print(f"alpha = {alphas_l[i]:7.3f}:  variables con peso distinto de 0 = {nz[i]:3d}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for j in range(10, P):
    ax[0].plot(alphas_l, coefs_l[j], color="#B0BEC5", lw=0.6)
for j in range(10):
    ax[0].plot(alphas_l, coefs_l[j], color="C0" if j < 6 else "C1", lw=1.4)
ax[0].set_xscale("log"); ax[0].set_xlabel("alpha"); ax[0].set_ylabel("coeficiente (estandarizado)")
ax[1].plot(alphas_l, nz, color="C2"); ax[1].set_xscale("log"); ax[1].set_xlabel("alpha"); ax[1].set_ylabel("variables activas")
plt.tight_layout(); plt.show()

In [ ]:
lasso = ...     # ✏️ make_pipeline(StandardScaler(), LassoCV(alphas=ALPHAS, cv=kf, max_iter=100000)).fit(Xtr, ytr)
coef_l = lasso[-1].coef_
reales = int((coef_l[:10] != 0).sum()); falsas = int((coef_l[10:] != 0).sum())
print(f"LassoCV eligió alpha = {lasso[-1].alpha_:.3f}")
print(f"Variables activas: {(coef_l != 0).sum()}   (reales halladas: {reales} de 10;  de ruido que se colaron: {falsas})")
print(f"RMSE entrenamiento = {rmse(lasso, Xtr, ytr):.2f}    RMSE prueba = {rmse(lasso, Xte, yte):.2f}")
print("Variables reales que Lasso dejó en 0:", [j for j in range(10) if coef_l[j] == 0])

¿Qué pasa si **subimos** $\alpha$ para quedarnos con un modelo más corto? Multiplicamos el $\alpha$ elegido por distintos factores (celda completa).

In [ ]:
a0 = lasso[-1].alpha_
filas = []
for mult in [0.25, 0.5, 1, 2, 4, 8]:
    m = make_pipeline(StandardScaler(), Lasso(alpha=a0 * mult, max_iter=100000)).fit(Xtr, ytr)
    c = m[-1].coef_
    filas.append({"alpha / alpha_CV": mult, "variables activas": int((c != 0).sum()), "reales halladas": int((c[:10] != 0).sum()),
                  "de ruido": int((c[10:] != 0).sum()), "RMSE prueba": round(rmse(m, Xte, yte), 2)})
print(pd.DataFrame(filas).to_string(index=False))

✏️ **Pregunta 4.1.** (a) ¿En qué se diferencia la trayectoria de Lasso de la de Ridge? (b) ¿Cuántas variables activas deja `LassoCV`, cuántas son reales y cuántas son ruido? (c) Al subir $\alpha$ a 4 veces el elegido, ¿qué ganas y qué pierdes? (d) ¿Por qué «seleccionada por Lasso» no significa «causa del resultado»?

**Respuesta:**

## Parte 5 — Elastic Net  *(presentación D1–D4)*

Elastic Net mezcla L1 y L2 con `l1_ratio` $=\rho$ ($\rho=1$ es Lasso). Buscamos $\alpha$ y $\rho$ **juntos** con `GridSearchCV`; los parámetros del pipeline se nombran `paso__parametro`.

In [ ]:
pipe_en = make_pipeline(StandardScaler(), ElasticNet(max_iter=100000))
malla = {"elasticnet__alpha": np.logspace(-1.5, 0.5, 14),
         "elasticnet__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0]}
bus = ...     # ✏️ GridSearchCV(pipe_en, malla, cv=kf, scoring="neg_root_mean_squared_error")
bus.fit(Xtr, ytr)
print("Mejores parámetros:", bus.best_params_)
print(f"RMSE de validación = {-bus.best_score_:.2f}    RMSE de prueba = {rmse(bus, Xte, yte):.2f}")

R = pd.DataFrame(-bus.cv_results_["mean_test_score"].reshape(len(malla["elasticnet__alpha"]), len(malla["elasticnet__l1_ratio"])).T,
                 index=malla["elasticnet__l1_ratio"], columns=np.round(malla["elasticnet__alpha"], 2))
print("\nMejor RMSE de validación por cada l1_ratio:"); print(R.min(axis=1).round(2).to_string())
fig, ax = plt.subplots(figsize=(6.5, 3.3))
im = ax.imshow(R.values, aspect="auto", origin="lower", cmap="viridis_r")
ax.set_xticks(range(0, 14, 2)); ax.set_xticklabels(R.columns[::2]); ax.set_yticks(range(7)); ax.set_yticklabels(R.index)
ax.set_xlabel("alpha"); ax.set_ylabel("l1_ratio"); ax.grid(False); plt.colorbar(im, label="RMSE de validación"); plt.tight_layout(); plt.show()

**Variables correlacionadas.** Fijamos `l1_ratio=0.5` (con $\alpha$ por validación cruzada) y comparamos los pesos de los dos grupos correlacionados ($x_1$–$x_3$, verdad $+2$; $x_4$–$x_6$, verdad $-1.5$) en los cuatro modelos.

In [ ]:
en05 = make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[0.5], alphas=ALPHAS, cv=kf, max_iter=100000)).fit(Xtr, ytr)
def en_original(m):          # pesos en las unidades originales (deshace el escalado)
    return m[-1].coef_ / m[0].scale_
grupo = pd.DataFrame({"verdad": BETA[:6],
                      "OLS": en_original(ols)[:6], "Ridge": en_original(ridge)[:6],
                      "Lasso": en_original(lasso)[:6], "ElasticNet(0.5)": en_original(en05)[:6]},
                     index=[f"x{j + 1}" for j in range(6)])
print(grupo.round(2).to_string())
print(f"\nVariables activas -> Lasso: {(lasso[-1].coef_ != 0).sum()}   Elastic Net (0.5): {(en05[-1].coef_ != 0).sum()}")
print(f"RMSE de prueba    -> Lasso: {rmse(lasso, Xte, yte):.2f}   Elastic Net (0.5): {rmse(en05, Xte, yte):.2f}")

✏️ **Pregunta 5.1.** (a) ¿Qué `l1_ratio` ganó en la malla y qué te dice sobre este problema (verdad dispersa)? (b) En los grupos correlacionados, ¿cómo reparte el peso Lasso y cómo Elastic Net (0.5)? (c) ¿Qué precio paga Elastic Net (0.5) en variables activas y error de prueba?

**Respuesta:**

## Parte 6 — Comparación de los cuatro modelos con el mismo protocolo  *(presentación E1–E3)*

Ponemos los cuatro en igualdad de condiciones: `StandardScaler` → modelo, con los hiperparámetros elegidos **solo con el entrenamiento** (validación cruzada de 5 particiones). El conjunto de prueba se usa para evaluar al final.

In [ ]:
modelos = {
    "OLS": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 50), cv=kf)),
    "Lasso": ...,          # ✏️ make_pipeline(StandardScaler(), LassoCV(alphas=ALPHAS, cv=kf, max_iter=100000))
    "Elastic Net": ...,    # ✏️ make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[.1, .3, .5, .7, .9, .95], alphas=ALPHAS, cv=kf, max_iter=100000))
}
filas, coefs_orig = [], {}
for nombre, m in modelos.items():
    m.fit(Xtr, ytr)
    c = m[-1].coef_; coefs_orig[nombre] = c / m[0].scale_
    hip = []
    if hasattr(m[-1], "alpha_"): hip.append(f"alpha={m[-1].alpha_:.3g}")
    if hasattr(m[-1], "l1_ratio_"): hip.append(f"l1_ratio={m[-1].l1_ratio_:g}")
    filas.append({"método": nombre, "hiperparámetros": ", ".join(hip) or "-",
                  "rmse_entrenamiento": round(rmse(m, Xtr, ytr), 2), "rmse_prueba": round(rmse(m, Xte, yte), 2),
                  "variables_usadas": int((np.abs(c) > 1e-8).sum()), "reales_halladas": int((np.abs(c[:10]) > 1e-8).sum())})
comparacion = pd.DataFrame(filas)
print(comparacion.to_string(index=False))

fig, ax = plt.subplots(2, 2, figsize=(10, 5), sharex=True, sharey=True)
for a_, (nombre, c) in zip(ax.ravel(), coefs_orig.items()):
    a_.vlines(np.arange(P), 0, c, lw=0.9); a_.plot(np.arange(10), BETA[:10], "ko", ms=3); a_.set_title(nombre, fontsize=9); a_.set_ylim(-5, 6)
for a_ in ax[1]: a_.set_xlabel("índice de variable")
plt.tight_layout(); plt.show()

✏️ **Pregunta 6.1.** (a) Ordena los cuatro modelos por RMSE de prueba y explica el RMSE de entrenamiento de OLS. (b) ¿Qué modelos usan menos variables y cuántas de las 10 reales encuentran? (c) Esta tabla viene de **una sola** muestra de 100 observaciones: ¿cuánta diferencia entre Lasso y Elastic Net te parecería real?

**Respuesta:**

In [ ]:
comentarios = {
    "OLS": "✏️ escribe tu comentario",
    "Ridge": "✏️ escribe tu comentario",
    "Lasso": "✏️ escribe tu comentario",
    "Elastic Net": "✏️ escribe tu comentario",
}
entregable = comparacion.copy()
entregable["comentario"] = entregable["método"].map(comentarios)
entregable.to_csv("entregable_unidad5.csv", index=False)
entregable[["método", "hiperparámetros", "rmse_prueba", "variables_usadas", "reales_halladas", "comentario"]]

## Parte 7 — ¿Y si todos los coeficientes importan un poco?  *(presentación E4)*

La función `compara` repite el experimento con 10 muestras nuevas (otro $n=100$ cada vez) y promedia el RMSE de prueba y el número de variables activas. Comparamos dos mundos: **verdad dispersa** (la de siempre) y **verdad densa** (los 150 coeficientes son distintos de cero y pequeños).

In [ ]:
def compara(beta, reps=10, n=100):
    acum = {k: {"rmse": [], "nz": []} for k in ["OLS", "Ridge", "Lasso", "Elastic Net"]}
    for r in range(reps):
        X1, y1, X2, y2 = genera(n, semilla=1000 + r, beta=beta)
        ms = {"OLS": make_pipeline(StandardScaler(), LinearRegression()),
              "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 50), cv=kf)),
              "Lasso": make_pipeline(StandardScaler(), LassoCV(alphas=ALPHAS, cv=kf, max_iter=100000)),
              "Elastic Net": make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[.1, .3, .5, .7, .9, .95], alphas=ALPHAS, cv=kf, max_iter=100000))}
        for k, m in ms.items():
            m.fit(X1, y1)
            acum[k]["rmse"].append(rmse(m, X2, y2)); acum[k]["nz"].append((np.abs(m[-1].coef_) > 1e-8).sum())
    return pd.DataFrame({k: {"RMSE prueba": np.mean(v["rmse"]), "desv. estándar": np.std(v["rmse"]), "variables activas": np.mean(v["nz"])}
                         for k, v in acum.items()}).T.round(2)


BETA_DENSO = np.random.default_rng(99).standard_normal(P) * 0.35
disperso = ...     # ✏️ compara(BETA)
denso = ...        # ✏️ compara(BETA_DENSO)
print("Verdad DISPERSA (10 de 150):"); print(disperso.to_string())
print("\nVerdad DENSA (150 de 150):"); print(denso.to_string())

fig, ax = plt.subplots(figsize=(6.5, 3.2)); x = np.arange(4)
ax.bar(x - 0.2, disperso["RMSE prueba"], 0.4, label="dispersa"); ax.bar(x + 0.2, denso["RMSE prueba"], 0.4, label="densa")
ax.axhline(SIGMA, color="k", ls=":", lw=1); ax.set_xticks(x); ax.set_xticklabels(disperso.index); ax.set_ylabel("RMSE de prueba"); ax.legend(); plt.tight_layout(); plt.show()

✏️ **Pregunta 7.1.** (a) ¿Qué método gana con verdad dispersa y cuál con verdad densa? (b) ¿Por qué Lasso pierde cuando todos los coeficientes aportan un poco? (c) ¿Qué método elegirías si **no** supieras cómo es la verdad y por qué?

**Respuesta:**

## Parte 8 — Recomendación final

✏️ **Pregunta 8.1.** Redacta una recomendación breve (6 a 8 líneas) para un colega que va a modelar un problema con muchas más variables que observaciones: qué método usarías y con qué pasos (estandarizar, validar, elegir hiperparámetros), qué RMSE de prueba esperar frente a OLS **en estos datos**, qué ventajas y límites tiene tu elección (falsos positivos, correlacionadas, verdad dispersa o densa) y qué cuidados tomaste para no engañarte (conjunto de prueba, escalado dentro del pipeline).

**Respuesta:**

---
**Reto opcional — ¿y si no estandarizo?** Multiplicamos cada columna por una escala distinta (entre 0.1 y 10) y comparamos Lasso **con** y **sin** `StandardScaler`. ¿Qué variables sobreviven en cada caso?

In [ ]:
escalas = np.exp(np.random.default_rng(5).uniform(np.log(0.1), np.log(10), P))     # escala aleatoria por columna
Xtr_e, Xte_e = Xtr * escalas, Xte * escalas
con = make_pipeline(StandardScaler(), LassoCV(alphas=ALPHAS, cv=kf, max_iter=100000)).fit(Xtr_e, ytr)
sin = LassoCV(alphas=np.logspace(-4, 1.2, 60), cv=kf, max_iter=100000).fit(Xtr_e, ytr)
for nombre, m in [("con StandardScaler", con), ("sin escalar", sin)]:
    c = m[-1].coef_ if hasattr(m, "steps") else m.coef_
    print(f"{nombre:20s} RMSE prueba = {rmse(m, Xte_e, yte):.2f}   activas = {(c != 0).sum():3d}   reales halladas = {(c[:10] != 0).sum()} de 10")
print("Escala de las 10 variables reales:", np.round(escalas[:10], 2))

✏️ **Pregunta del reto.** ¿Qué diferencia hay entre el modelo con y sin escalado? ¿Qué variables reales sobreviven en cada caso y por qué la escala de una variable decide su suerte en Lasso?

**Respuesta:**